# Customer Churn Prediction & Machine Learning Modeling

Trains and benchmarks supervised classification algorithms (Random Forest, Gradient Boosting, Logistic Regression) on customer RFM features.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


## 1. Load Cleaned Customer Features


In [ ]:
cust_path = Path('../data/raw/customers.csv')
orders_path = Path('../data/raw/orders.csv')

df_cust = pd.read_csv(cust_path)
df_orders = pd.read_csv(orders_path)
print(f"Loaded {len(df_cust):,} customers and {len(df_orders):,} orders.")


## 2. Feature Engineering & RFM Calculation


In [ ]:
ref_date = pd.to_datetime('2026-08-01')
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

agg = df_orders.groupby('customer_id').agg(
    total_orders=('order_id', 'count'),
    total_spend=('total_amount', 'sum'),
    avg_order_value=('total_amount', 'mean'),
    last_order=('order_date', 'max')
).reset_index()

agg['recency_days'] = (ref_date - agg['last_order']).dt.days

df_ml = df_cust.merge(agg, on='customer_id', how='left').fillna(0)
df_ml['churn'] = ((df_ml['recency_days'] > 180) | ((df_ml['recency_days'] > 120) & (df_ml['total_orders'] <= 1))).astype(int)

features = ['recency_days', 'total_orders', 'total_spend', 'avg_order_value']
X = df_ml[features]
y = df_ml['churn']
print(f"Target Class Distribution:\n{y.value_counts(normalize=True)}")


## 3. Model Training & Evaluation


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


## 4. Confusion Matrix & ROC Curve Visualization


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'])
plt.title('Random Forest Confusion Matrix', fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()


## 5. Feature Importances


In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette='Blues_r')
plt.title('Feature Importances in Churn Classification', fontweight='bold')
plt.xlabel('Importance')
plt.show()
